In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_raw = pd.read_csv('kidney_disease.csv')
print(f"Shape: {df_raw.shape}  →  {df_raw.shape[0]} records, {df_raw.shape[1]} features")
pd.set_option('display.max_columns', None)
df_raw.head()

Shape: (400, 26)  →  400 records, 26 features


,id,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,bu,sc,sod,pot,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,36.0,1.2,NaN,NaN,15.4,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,18.0,0.8,NaN,NaN,11.3,38,6000,NaN,no,no,no,good,no,no,ckd
2,2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,53.0,1.8,NaN,NaN,9.6,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,56.0,3.8,111.0,2.5,11.2,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,26.0,1.4,NaN,NaN,11.6,35,7300,4.6,no,no,no,good,no,no,ckd


In [3]:
# ── 3A. Class Balance (MUST be checked early) ──────────────────────────────
print("=== Target Class Distribution ===\n", df_raw['classification'].value_counts(), "\n")

# ── 3B. Column-level quality overview ──────────────────────────────────────
def scan_data_quality(df):
    hidden_numeric, dirty_categorical = [], []
    for col in df.select_dtypes(include=['object', 'string']).columns:
        vals = df[col].dropna().unique()
        if vals.size == 0: continue
        
        if pd.to_numeric(pd.Series(vals), errors='coerce').notna().mean() > 0.5 and df[col].nunique() != len(df):
            hidden_numeric.append(col)
        elif vals.size != pd.Series(vals).astype(str).str.strip().nunique():
            dirty_categorical.append(col)
    return hidden_numeric, dirty_categorical

hidden_numeric_cols, dirty_cols_names = scan_data_quality(df_raw)

# Generate Quality Table
col_info = pd.DataFrame({
    'pandas_dtype' : df_raw.dtypes.astype(str),
    'null_count'   : df_raw.isna().sum(),
    'null_%'       : (df_raw.isna().mean() * 100).round(2),
    'n_unique'     : df_raw.nunique(),
    'sample_values': [str(df_raw[c].dropna().unique()[:3].tolist()) for c in df_raw.columns]
}).sort_values('null_%', ascending=False)

print("=== Column Quality Scan ===")
# Inline Row Styling based on row index name
display(col_info.style.apply(lambda row: 
    ['background-color: #ffe6e6'] * len(row) if row.name in hidden_numeric_cols else 
    ['background-color: #fff2cc'] * len(row) if row.name in dirty_cols_names else 
    [''] * len(row), axis=1))

if hidden_numeric_cols: print(f"• Hidden Numeric Columns (Red): {hidden_numeric_cols}")
if dirty_cols_names: print(f"• Dirty Categorical Columns (Yellow): {dirty_cols_names}")
if not hidden_numeric_cols and not dirty_cols_names:print("Wrong data types and dirty entries have been corrected.")

=== Target Class Distribution ===
 classification
ckd       248
notckd    150
ckd\t       2
Name: count, dtype: int64 

=== Column Quality Scan ===


,pandas_dtype,null_count,null_%,n_unique,sample_values
rbc,object,152,38.000000,2,"['normal', 'abnormal']"
rc,object,130,32.500000,49,"['5.2', '3.9', '4.6']"
wc,object,105,26.250000,92,"['7800', '6000', '7500']"
pot,float64,88,22.000000,40,"[2.5, 3.2, 4.0]"
sod,float64,87,21.750000,34,"[111.0, 142.0, 104.0]"
pcv,object,70,17.500000,44,"['44', '38', '31']"
pc,object,65,16.250000,2,"['normal', 'abnormal']"
hemo,float64,52,13.000000,115,"[15.4, 11.3, 9.6]"
su,float64,49,12.250000,6,"[0.0, 3.0, 4.0]"
sg,float64,47,11.750000,5,"[1.02, 1.01, 1.005]"


• Hidden Numeric Columns (Red): ['pcv', 'wc', 'rc']
• Dirty Categorical Columns (Yellow): ['dm', 'cad', 'classification']


In [4]:
df_clean = df_raw.copy()
print("--- Starting Automated Data Cleaning Pipeline ---")

# STEP 1: Global String Clean & Special Character Replacement
# Doing this first fixes 'ckd\t' and '?' globally in one fast pass
print("• Standardizing all text data and handling missing value placeholders...")
for col in df_clean.select_dtypes(include=['object', 'string']).columns:
    # Vectorized strip is much faster than .apply(lambda x: ...)
    df_clean[col] = df_clean[col].astype(str).str.strip()
    
    # Replace common placeholder strings with true NaNs
    df_clean[col] = df_clean[col].replace({'?': np.nan, 'nan': np.nan, 'None': np.nan})

# STEP 2: Safe Downcasting for Hidden Numeric Columns
# Since whitespace and '?' are already handled above, we just convert data types
if hidden_numeric_cols:
    for col in hidden_numeric_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    print(f"✓ Converted hidden numeric columns to numeric: {hidden_numeric_cols}")
else:
    print("• No hidden numeric columns to convert.")

print("✓ Pipeline complete. All text columns stripped and standardized.")
print(f"\nTarget unique values after cleaning: {df_clean['classification'].dropna().unique()}")

--- Starting Automated Data Cleaning Pipeline ---
• Standardizing all text data and handling missing value placeholders...
✓ Converted hidden numeric columns to numeric: ['pcv', 'wc', 'rc']
✓ Pipeline complete. All text columns stripped and standardized.

Target unique values after cleaning: ['ckd' 'notckd']


In [5]:
df_clean = df_clean.drop('id', axis=1)
df_clean.shape

(400, 25)

In [6]:
df=df_clean.copy()
# ── 3A. Class Balance (MUST be checked early) ──────────────────────────────
print("=== Target Class Distribution ===\n", df['classification'].value_counts(), "\n")

# ── 3B. Column-level quality overview ──────────────────────────────────────
def scan_data_quality(df):
    hidden_numeric, dirty_categorical = [], []
    for col in df.select_dtypes(include=['object', 'string']).columns:
        vals = df[col].dropna().unique()
        if vals.size == 0: continue
        
        if pd.to_numeric(pd.Series(vals), errors='coerce').notna().mean() > 0.5 and df[col].nunique() != len(df):
            hidden_numeric.append(col)
        elif vals.size != pd.Series(vals).astype(str).str.strip().nunique():
            dirty_categorical.append(col)
    return hidden_numeric, dirty_categorical

hidden_numeric_cols, dirty_cols_names = scan_data_quality(df)

# Generate Quality Table
col_info = pd.DataFrame({
    'pandas_dtype' : df.dtypes.astype(str),
    'null_count'   : df.isna().sum(),
    'null_%'       : (df.isna().mean() * 100).round(2),
    'n_unique'     : df.nunique(),
    'sample_values': [str(df[c].dropna().unique()[:3].tolist()) for c in df.columns]
}).sort_values('null_%', ascending=False)

print("=== Column Quality Scan ===")
# Inline Row Styling based on row index name
display(col_info.style.apply(lambda row: 
    ['background-color: #ffe6e6'] * len(row) if row.name in hidden_numeric_cols else 
    ['background-color: #fff2cc'] * len(row) if row.name in dirty_cols_names else 
    [''] * len(row), axis=1))

# Print summary alerts cleanly
if hidden_numeric_cols:print(f"• Hidden Numeric Columns (Red): {hidden_numeric_cols}")
if dirty_cols_names:print(f"• Dirty Categorical Columns (Yellow): {dirty_cols_names}")
if not hidden_numeric_cols and not dirty_cols_names:print("Wrong data types and dirty entries have been corrected.")

=== Target Class Distribution ===
 classification
ckd       250
notckd    150
Name: count, dtype: int64 

=== Column Quality Scan ===


,pandas_dtype,null_count,null_%,n_unique,sample_values
rbc,object,152,38.000000,2,"['normal', 'abnormal']"
rc,float64,131,32.750000,45,"[5.2, 3.9, 4.6]"
wc,float64,106,26.500000,89,"[7800.0, 6000.0, 7500.0]"
pot,float64,88,22.000000,40,"[2.5, 3.2, 4.0]"
sod,float64,87,21.750000,34,"[111.0, 142.0, 104.0]"
pcv,float64,71,17.750000,42,"[44.0, 38.0, 31.0]"
pc,object,65,16.250000,2,"['normal', 'abnormal']"
hemo,float64,52,13.000000,115,"[15.4, 11.3, 9.6]"
su,float64,49,12.250000,6,"[0.0, 3.0, 4.0]"
sg,float64,47,11.750000,5,"[1.02, 1.01, 1.005]"


Wrong data types and dirty entries have been corrected.


In [7]:
print("--- Checking for Duplicate Rows ---")
duplicate_row_count = df.duplicated().sum()

if duplicate_row_count > 0:
    df = df.drop_duplicates(keep='first').reset_index(drop=True)
    print(f"✓ Removed {duplicate_row_count} duplicate rows.")
else:
    print("✓ Clean! No duplicate rows detected.")

print(f"Current Shape: {df.shape[0]} rows, {df.shape[1]} features\n")


# --- 2. Lightning-Fast Duplicate Column Investigation ---
print("--- Investigating Duplicate Columns ---")

# Vectorized transpose detection finds identical columns in C-speed, not Python loops
is_duplicate_col = df.T.duplicated(keep='first')

if is_duplicate_col.any():
    # Grab the names of the duplicate columns to be removed
    duplicate_cols = df.columns[is_duplicate_col].tolist()
    print(f"⚠ Found {len(duplicate_cols)} identical duplicate column(s): {duplicate_cols}")
    
    # Optional: Map duplicates to their original columns for clear logging
    for col in duplicate_cols:
        # Find the first column in the dataframe that matches this duplicate column perfectly
        original = df.columns[(df.T == df[col]).all(axis=1)][0]
        print(f"  • Column '{col}' is an identical twin of original column: '{original}'")
        
    print("\nℹ Action: No columns were deleted yet. Please review the twins listed above.")
else:
    print("✓ Clean! No duplicate columns detected.")

print(f"Final Shape: {df.shape[0]} rows, {df.shape[1]} features\n")

--- Checking for Duplicate Rows ---
✓ Clean! No duplicate rows detected.
Current Shape: 400 rows, 25 features

--- Investigating Duplicate Columns ---
✓ Clean! No duplicate columns detected.
Final Shape: 400 rows, 25 features



In [8]:
print("Before encoding:", df['classification'].unique())
target_mapping = {'ckd': 1, 'notckd': 0}
df['classification'] = df['classification'].map(target_mapping).astype(int)
print("After encoding: ", df['classification'].unique())
print("Class counts:\n", df['classification'].value_counts())

Before encoding: ['ckd' 'notckd']
After encoding:  [1 0]
Class counts:
 classification
1    250
0    150
Name: count, dtype: int64


In [9]:
# Save the DataFrame to a CSV file without the index column
df.to_csv('preprocessed_kidney_data.csv', index=False)
print("DataFrame saved successfully!")

DataFrame saved successfully!
